In [ ]:
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'common.ipynb').is_file()))
get_ipython().run_line_magic('run', '-i "' + str(ROOT / 'src' / 'common.ipynb') + '"')

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
FIG = OUT / 'figures'
plt.rcdefaults()
plt.rcParams.update({'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42})
import shap
np.random.seed(SEED)


In [ ]:
all_shap_values, all_X_test_renamed, all_X = ({}, {}, {})
importance = []
for code in MODELS:
    fitted = fit_global(code)
    data, test, model = fitted['data'], fitted['test_indices'], fitted['model']
    X = data.iloc[test][FEATURES].reset_index(drop=True)
    contributions = model.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)
    values = contributions[:, :-1].astype(np.float64)
    bias = contributions[:, -1].astype(np.float64)
    assert np.max(np.abs(values.sum(axis=1) + bias - model.predict(X))) < 0.0001
    absolute = np.abs(values).mean(axis=0)
    for feature, value in zip(FEATURES, absolute):
        importance.append({'model': code, 'feature': feature, 'label': LABELS[feature], 'mean_abs_SHAP_pp': value, 'relative_importance_pct': 100 * value / absolute.sum()})
    all_shap_values[code] = values
    all_X[code] = X
    all_X_test_renamed[code] = X.rename(columns=LABELS)
importance = pd.DataFrame(importance)


In [ ]:
ranking = importance.loc[importance['model'].eq('ALL')].sort_values('mean_abs_SHAP_pp', ascending=False)['feature'].tolist()
column_names = {'ALL': 'Overall', **SEASON_NAMES}
for number, measure in [(5, 'mean_abs_SHAP_pp'), (6, 'relative_importance_pct')]:
    table = importance.pivot(index='feature', columns='model', values=measure).loc[ranking, MODELS]
    table.index = [LABELS[f] for f in table.index]
    table.index.name = 'Feature'
    table = table.rename(columns=column_names).reset_index()
    export_table(f'TableS{number}', table.round(2))


In [ ]:
def export_figure(fig, stem, performance=False):
    for ax in fig.axes:
        for collection in ax.collections:
            collection.set_rasterized(False)
        if performance or ax.get_xlabel() in ['mean(|SHAP|)', 'SHAP value']:
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        if performance:
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    fig.canvas.draw()
    bounds = {} if stem == 'FigureS3' else {'bbox_inches': 'tight'}
    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(FIG / f'{stem}.{ext}', dpi=300, **bounds)
    plt.close(fig)


In [ ]:
season_order = SEASONS
predictors_mapping = LABELS


In [ ]:
sub_labels = ['a', 'b', 'c', 'd']
fig, axs = plt.subplots(4, 2, figsize=(20, 20), gridspec_kw={'width_ratios': [1.0, 1.5]}, sharey='row')
for i, season in enumerate(season_order):
    s_val = all_shap_values[season]
    s_X = all_X_test_renamed[season]
    max_display = min(20, s_val.shape[1])
    mean_abs = np.abs(s_val).mean(axis=0)
    order_idx = np.argsort(mean_abs)[::-1][:max_display]
    s_val_plot = s_val[:, order_idx]
    s_X_plot = s_X.iloc[:, order_idx]
    ax_bar = axs[i, 0]
    plt.sca(ax_bar)
    shap.summary_plot(s_val_plot, s_X_plot, plot_type='bar', show=False, sort=False, max_display=max_display, plot_size=None)
    ax_bar.set_title('')
    ax_bar.set_xlabel('mean(|SHAP|)')
    for p in ax_bar.patches:
        p.set_facecolor('steelblue')
    ax_bar.invert_xaxis()
    ax_bar.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_bar.text(0, 1.03, sub_labels[i], transform=ax_bar.transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
    ax_dot = axs[i, 1]
    plt.sca(ax_dot)
    shap.summary_plot(s_val_plot, s_X_plot, show=False, plot_type='dot', sort=False, max_display=max_display, plot_size=None)
    ax_dot.set_title('')
    ax_dot.set_xlabel('SHAP value')
    ax_dot.tick_params(axis='y', which='both', labelleft=True)
    for t in ax_dot.get_yticklabels():
        t.set_ha('center')
        t.set_x(-0.3)
    for a in (ax_bar, ax_dot):
        for spine in a.spines.values():
            spine.set_visible(False)
fig.subplots_adjust(left=0.06, right=0.99, top=0.99, bottom=0.03, hspace=0.22, wspace=0.7)
export_figure(fig, 'Figure3')


In [ ]:
X_test = all_X['ALL']
shap_vals = all_shap_values['ALL']
mean_abs_shap = np.abs(shap_vals).mean(axis=0)


In [ ]:
X_test_named = X_test.rename(columns=predictors_mapping)
max_display = 20
order_idx = np.argsort(mean_abs_shap)[::-1]
if len(order_idx) > max_display:
    order_idx = order_idx[:max_display]
shap_vals_plot = shap_vals[:, order_idx]
X_test_named_plot = X_test_named.iloc[:, order_idx]
fig, (ax_bar, ax_dot) = plt.subplots(1, 2, figsize=(20, 6), sharey=True, gridspec_kw={'width_ratios': [1.0, 1.5], 'wspace': 0.7})
plt.sca(ax_bar)
shap.summary_plot(shap_vals_plot, X_test_named_plot, show=False, plot_type='bar', sort=False, max_display=max_display, plot_size=None)
ax_bar.set_title('')
ax_bar.set_xlabel('mean(|SHAP|)')
for p in ax_bar.patches:
    p.set_facecolor('steelblue')
ax_bar.invert_xaxis()
ax_bar.tick_params(axis='y', which='both', left=False, labelleft=False)
plt.sca(ax_dot)
shap.summary_plot(shap_vals_plot, X_test_named_plot, show=False, plot_type='dot', sort=False, max_display=max_display, plot_size=None)
ax_dot.set_title('')
ax_dot.set_xlabel('SHAP value')
ax_dot.tick_params(axis='y', which='both', labelleft=True)
for t in ax_dot.get_yticklabels():
    t.set_ha('center')
    t.set_x(-0.3)
for a in (ax_bar, ax_dot):
    for spine in a.spines.values():
        spine.set_visible(False)
fig.subplots_adjust(left=0.01, right=0.999, top=0.97, bottom=0.1, wspace=0.1)
export_figure(fig, 'FigureS3')


In [ ]:
records = []
for season in SEASONS:
    values = all_shap_values[season]
    prediction = values.sum(axis=1)
    centered = prediction - prediction.mean()
    variance = np.mean(centered ** 2)
    covariance = np.mean((values - values.mean(axis=0)) * centered[:, None], axis=0)
    share = 100 * covariance / variance
    assert abs(share.sum() - 100) < 1e-10
    for feature, value in zip(FEATURES, share):
        records.append({'season': season, 'feature': feature, 'share_of_prediction_variance_pct': value})
variance_table = pd.DataFrame(records)
pivot = variance_table.pivot(index='feature', columns='season', values='share_of_prediction_variance_pct')
order = pivot[SEASONS].abs().mean(axis=1).sort_values(ascending=False).index
features = order[:10].tolist()
top = pivot.loc[features, SEASONS].copy()
top.index = [LABELS[f] for f in features]
COLORS = {'SP': '#338535', 'SU': '#AD352C', 'FA': '#D48104', 'WI': '#297BBD'}
AXIS_LABEL = 'Share of variance in model predictions (%)'
SEASON_LABELS = SEASON_NAMES


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={'width_ratios': [1, 1]})
ax = axes[0]
group_w = 0.8
bar_w2 = group_w / len(SEASONS)
for s_i, season in enumerate(SEASONS):
    offsets = np.arange(len(top)) + s_i * bar_w2 - group_w / 2
    ax.barh(offsets, top[season], height=bar_w2 * 0.9, color=COLORS[season], label=SEASON_LABELS[season], zorder=2)
ax.set_yticks(np.arange(len(top)))
ax.set_yticklabels(top.index, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel(AXIS_LABEL)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(title='Season', loc='lower left', bbox_to_anchor=(0.92, 0))
for spine in ax.spines.values():
    spine.set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax2 = axes[1]
heat_data = top[SEASONS]
im = ax2.imshow(heat_data.values, cmap='RdYlGn_r', aspect='auto', vmin=heat_data.values.min(), vmax=heat_data.values.max())
ax2.set_xticks(range(len(SEASONS)))
ax2.set_xticklabels(SEASONS, fontsize=10)
ax2.set_yticks(range(len(top)))
ax2.set_yticklabels(top.index, fontsize=9)
for i in range(len(top)):
    for j, season in enumerate(SEASONS):
        val = heat_data.values[i, j]
        ax2.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color='black')
cb = plt.colorbar(im, ax=ax2, label=AXIS_LABEL, shrink=0.8)
cb.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.text(-0.08, 1.0, 'a', transform=ax.transAxes, fontsize=18, fontweight='bold', ha='left', va='bottom')
ax2.text(-0.08, 1.0, 'b', transform=ax2.transAxes, fontsize=18, fontweight='bold', ha='left', va='bottom')
plt.tight_layout()
export_figure(fig, 'Figure4')
